In [ ]:
# cell 1
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
from scipy import stats

DATA = r"D:\2026 Work\My Papers\waste glass concrete\Revision of Manuscript\1st Revision\Work\WGP_revision\01_WGP_database_with_ids.csv"
OUT = os.path.join(os.path.dirname(DATA), "Fig1_Fig2")
os.makedirs(OUT, exist_ok=True)
df = pd.read_csv(DATA)
FEATS = ["GGP size", "Replacement", "Cement", "Max size", "CA", "FA", "WC Ratio", "SiO2", "CaO", "Na2O", "Days"]
AXL = {"GGP size": "WGP particle size (\u03bcm)", "Replacement": "WGP replacement level (%)", "Cement": "Cement content (kg/m\u00b3)",
       "Max size": "Maximum aggregate size (mm)", "CA": "Coarse aggregate (kg/m\u00b3)", "FA": "Fine aggregate (kg/m\u00b3)",
       "WC Ratio": "Water-to-cement ratio (-)", "SiO2": "SiO\u2082 in WGP (%)", "CaO": "CaO in WGP (%)",
       "Na2O": "Na\u2082O in WGP (%)", "Days": "Curing age (days)"}
YL = "Compressive strength (MPa)"
ACC = "#B5525C"
plt.rcParams.update({"font.family": "Times New Roman", "font.size": 12, "axes.linewidth": 1.1, "savefig.dpi": 600})
order = sorted(df["source_id"].unique())
cmap = matplotlib.colormaps["tab20"]
colors = {s: cmap(i / 20) for i, s in enumerate(order)}
markers = {s: "os^Dv<>p"[i % 8] for i, s in enumerate(order)}
names = dict(zip(df["source_id"], df["source"]))
df.shape

# cell 2
def kde_curve(v, lo, hi):
    v = np.asarray(v, float)
    if np.unique(v).size < 2:
        return None, None
    grid = np.linspace(lo, hi, 300)
    return grid, stats.gaussian_kde(v)(grid)


def marginal(ax, v, lo, hi, vertical=False):
    bins = np.linspace(lo, hi, 26)
    ax.hist(v, bins=bins, density=True, color=ACC, alpha=0.30, edgecolor="0.35", linewidth=0.5,
            orientation="horizontal" if vertical else "vertical")
    g, d = kde_curve(v, lo, hi)
    if g is not None:
        if vertical:
            ax.fill_betweenx(g, 0, d, color=ACC, alpha=0.45)
            ax.plot(d, g, color=ACC, lw=2.2)
        else:
            ax.fill_between(g, 0, d, color=ACC, alpha=0.45)
            ax.plot(g, d, color=ACC, lw=2.2)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)


def joint(sf, c, label=None):
    gs = sf.add_gridspec(2, 2, width_ratios=[4.2, 1], height_ratios=[1, 4.2], hspace=0.04, wspace=0.04,
                         left=0.17, right=0.97, bottom=0.17, top=0.97)
    ax = sf.add_subplot(gs[1, 0])
    axx = sf.add_subplot(gs[0, 0], sharex=ax)
    axy = sf.add_subplot(gs[1, 1], sharey=ax)
    x, y = df[c].to_numpy(float), df["CS"].to_numpy(float)
    xp = (x.max() - x.min()) * 0.06
    xlo, xhi = x.min() - xp, x.max() + xp
    ylo, yhi = -5, 80
    for s in order:
        m = df["source_id"].to_numpy() == s
        ax.scatter(x[m], y[m], s=30, color=colors[s], marker=markers[s], edgecolor="0.15", linewidth=0.45, alpha=0.9, zorder=3)
    b1, b0 = np.polyfit(x, y, 1)
    xx = np.linspace(x.min(), x.max(), 200)
    yy = b0 + b1 * xx
    keep = (yy >= ylo) & (yy <= yhi)
    ax.plot(xx[keep], yy[keep], ls="--", lw=2.6, color=ACC, zorder=4)
    r = stats.pearsonr(x, y)[0]
    rho = stats.spearmanr(x, y)[0]
    ax.text(0.04, 0.96, f"Pearson r = {r:.3f}\nSpearman \u03c1 = {rho:.3f}", transform=ax.transAxes, va="top",
            fontsize=11.5, fontweight="bold", zorder=5,
            bbox=dict(facecolor="white", edgecolor=ACC, linewidth=1.6, boxstyle="round,pad=0.35"))
    ax.text(0.97, 0.04, f"n = {len(x)}", transform=ax.transAxes, ha="right", va="bottom", fontsize=10.5,
            fontweight="bold", color=ACC, zorder=5, bbox=dict(facecolor="white", edgecolor=ACC, linewidth=1.3, boxstyle="round,pad=0.25"))
    ax.set_xlim(xlo, xhi)
    ax.set_ylim(ylo, yhi)
    ax.grid(ls="--", color="0.82", lw=0.7, zorder=0)
    ax.set_xlabel(AXL[c], fontsize=13, fontweight="bold")
    ax.set_ylabel(YL, fontsize=13, fontweight="bold")
    marginal(axx, x, xlo, xhi)
    marginal(axy, y, ylo, yhi, vertical=True)
    axx.set_ylabel("Density", fontsize=10)
    axy.set_xlabel("Density", fontsize=10)
    axx.tick_params(labelbottom=False, labelsize=9)
    axy.tick_params(labelleft=False, labelsize=9)
    axx.yaxis.set_major_locator(MaxNLocator(2, prune="lower"))
    axy.xaxis.set_major_locator(MaxNLocator(2, prune="lower"))
    if label:
        sf.text(0.57, 0.005, label, ha="center", va="bottom", fontsize=14, fontweight="bold")
    return ax


handles = [Line2D([], [], ls="", marker=markers[s], markersize=8, markerfacecolor=colors[s], markeredgecolor="0.15",
                  markeredgewidth=0.5, label=f"{s}  {names[s]}") for s in order]
handles.append(Line2D([], [], ls="--", lw=2.6, color=ACC, label="Linear fit (all records)"))
len(handles)

# cell 3
fig = plt.figure(figsize=(16, 21.5))
subs = fig.subfigures(4, 3, wspace=0.02, hspace=0.02)
for i, (sf, c) in enumerate(zip(subs.flat, FEATS)):
    joint(sf, c, f"({chr(97 + i)})")
leg_ax = subs.flat[-1].add_subplot(111)
leg_ax.axis("off")
leg_ax.legend(handles=handles, loc="center", frameon=False, fontsize=11.5, handletextpad=0.6, labelspacing=0.55)
fig.savefig(os.path.join(OUT, "Fig1_bivariate_jointplots.png"), bbox_inches="tight", dpi=400)
fig.savefig(os.path.join(OUT, "Fig1_bivariate_jointplots.pdf"), bbox_inches="tight")
fig

# cell 4
for i, c in enumerate(FEATS):
    f = plt.figure(figsize=(6.2, 6.2))
    joint(f, c)
    f.savefig(os.path.join(OUT, f"Fig1{chr(97 + i)}_{c.replace(' ', '_')}.png"), bbox_inches="tight")
    plt.close(f)
lf = plt.figure(figsize=(4.6, 6.4))
lf.legend(handles=handles, loc="center", frameon=False, fontsize=11.5)
lf.savefig(os.path.join(OUT, "Fig1_legend.png"), bbox_inches="tight")
plt.close(lf)
sorted(os.listdir(OUT))

# cell 5
short = {"GGP size": "WGP size (\u03bcm)", "Replacement": "Replacement (%)", "Cement": "Cement (kg/m\u00b3)", "Max size": "Max size (mm)",
         "CA": "CA (kg/m\u00b3)", "FA": "FA (kg/m\u00b3)", "WC Ratio": "W/C (-)", "SiO2": "SiO\u2082 (%)", "CaO": "CaO (%)",
         "Na2O": "Na\u2082O (%)", "Days": "Curing age (d)", "CS": "CS (MPa)"}
corr = df[FEATS + ["CS"]].corr()
mask = np.triu(np.ones_like(corr, bool), k=1)
fig2, ax2 = plt.subplots(figsize=(11, 9.5))
im = ax2.imshow(np.ma.masked_where(mask, corr.values), cmap="RdBu_r", vmin=-1, vmax=1)
for i in range(len(corr)):
    for j in range(i + 1):
        v = corr.values[i, j]
        ax2.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=12, fontweight="bold" if abs(v) >= 0.4 and i != j else "normal",
                 color="white" if abs(v) > 0.55 else "black")
ticks = [short[c] for c in corr.columns]
ax2.set_xticks(range(len(ticks)))
ax2.set_xticklabels(ticks, rotation=45, ha="right", fontsize=13, fontweight="bold")
ax2.set_yticks(range(len(ticks)))
ax2.set_yticklabels(ticks, fontsize=13, fontweight="bold")
ax2.set_xticks(np.arange(-0.5, len(ticks)), minor=True)
ax2.set_yticks(np.arange(-0.5, len(ticks)), minor=True)
ax2.grid(which="minor", color="white", linewidth=1.5)
ax2.tick_params(which="minor", length=0)
for sp in ax2.spines.values():
    sp.set_visible(False)
cb = fig2.colorbar(im, ax=ax2, fraction=0.046, pad=0.03)
cb.set_label("Pearson correlation coefficient (r)", fontsize=13, fontweight="bold")
cb.ax.tick_params(labelsize=11)
fig2.tight_layout()
fig2.savefig(os.path.join(OUT, "Fig2_correlation_matrix.png"), bbox_inches="tight")
fig2.savefig(os.path.join(OUT, "Fig2_correlation_matrix.pdf"), bbox_inches="tight")
fig2